# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the **Refresh / Content Opportunity Scoring** lane as a concrete machine learning task.

## 1. My lane as an ML task (type)

I am framing this as a **Ranking and Scoring** task.

**Why:** The business goal isn't just to find every page that *might* be declining, but to tell the content team which pages they should review **first**. By producing a continuous score (0.0 to 1.0) and ranking the output, we provide a prioritized queue that respects the limited time of human editors.

In [1]:
print("Task: Ranking/Scoring")
print("Goal: Prioritized review queue")

Task: Ranking/Scoring
Goal: Prioritized review queue


## 2. Target or proxy

**Target:** Observed search performance decline.

**Proxy:** In this starter phase, I am using the `trend_direction == 'down'` flag as a proxy label. This is an **observed outcome** (calculated from real traffic changes) rather than a human-defined rule. 

In a more advanced version (using the full warehouse), I would define the target as a **future-window decline**: for example, a page that loses more than 20% of its click volume in the 30 days *after* the prediction date.

In [2]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
positive_cases = (df['trend_direction'] == 'down').sum()
print(f"Target Proxy: trend_direction == 'down'")
print(f"Positive examples in starter data: {positive_cases}")

Target Proxy: trend_direction == 'down'
Positive examples in starter data: 16262


## 3. Success metric

The primary success metric is **Precision@K** (specifically Precision@50).

**Why:** A content editor might only have time to review 50 pages a week. Success means that as many of those top 50 as possible are "true" opportunities (e.g., they actually show decline or significant CTR gaps). I will also track **Average Precision** to evaluate the quality of the ranking across the entire dataset.

In [3]:
print("Success Metric: Precision@50")
print("Baseline goal: Beat rule-based precision (~0.24)")

Success Metric: Precision@50
Baseline goal: Beat rule-based precision (~0.24)


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One row = One unique content item (page) for a specific client.

In [4]:
cols_to_show = ['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'trend_direction']
print(f"Unit of Analysis: Content Item (Page)")
print(f"Data Grain: client_id + content_id")
print(f"Shape: {df.shape}\n")
print("Example Rows:")
print(df[cols_to_show].head(3))

Unit of Analysis: Content Item (Page)
Data Grain: client_id + content_id
Shape: (30000, 44)

Example Rows:
             content_id          client_id  impressions_90d   ctr  \
0  content_304f48230142  client_f369cb89fc             3803  0.76   
1  content_a1fb4e703a9e  client_4e07408562            15320  0.05   
2  content_9aa793d4d895  client_7f2253d7e2            12581  0.09   

   avg_position trend_direction  
0          10.6            down  
1          20.3            down  
2          36.5            down  


## 5. Why ML beats a fixed rule here

A fixed rule (like "refresh if age > 180 days") is too blunt because:
1. **Signal Interaction:** A page might be old but still performing exceptionally well, or fresh but failing due to a low CTR. ML can find the non-linear relationship between age, position, and engagement.
2. **Scale and Complexity:** With 40+ signals including search volume, word count, and intent, a human cannot write a balanced weighting system by hand that stays accurate across different clients and content types.
3. **Dynamic Baselines:** What counts as a "low" CTR depends on the average position. ML can naturally learn these position-based expectations better than a nested if-statement.

In [5]:
print("Why ML: Non-linear interactions between 40+ signals.")

Why ML: Non-linear interactions between 40+ signals.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.